# Lab 00 · Rules, then LLM calls

**Prepared by Mustafa Assaf** · Lecture 1: Ask, check, retrieve · 15–20 minute lab

## One question, two ways to answer it

**Start here.** Every lab in this course answers the same question: should a team build a **modular monolith** or **microservices**? The question stays the same. Each lab adds one new technique.

We use one fictional course policy throughout:

> Choose **microservices** only when the team has at least 8 people **and** services must deploy independently. Otherwise choose a **modular monolith**.

This is a teaching rule, not real architecture advice. In this lab you will:

1. write the policy as a Python function that also explains its answer,
2. run it on four projects, one for each combination of the two facts,
3. write a helper that turns a free-text model reply into a label,
4. ask a model about the same four projects and count how often it agrees with the rule.

**At a glance**

![Lab 00 at a glance: 4 projects (every fact combination), then Apply the rule (decide()) or Ask a model (ask() + to_label()), then Compare (agreement count).](https://mkassaf.github.io/agentic-ai-course/assets/diagrams/00.svg)

## The idea in context

For decades, software made decisions with rules that people wrote by hand: if this, then that. The expert systems of the 1970s and 1980s stored hundreds of such rules, for example to suggest treatments for infections or to configure computer orders. Rules are exact, fast, and easy to audit. But every case must be foreseen, and someone must write and maintain every rule.

Machine learning changed where a decision comes from: instead of writing the rule, we show a model many examples, and it learns the pattern. Large language models (LLMs) go further. Trained on a vast amount of text, they can answer questions that nobody wrote a rule for, in plain language. The price is that their answers are plausible, not guaranteed, and an LLM does not know your organisation's policy unless you tell it.

This course keeps both. The rule is a reference that we can check a model against, and the model handles what a rule cannot express. Lab 00 puts them side by side on one question.

| Key term | Meaning |
| --- | --- |
| **Rule-based system** | A program whose decisions come from rules that people wrote explicitly. |
| **Large language model (LLM)** | A model trained on large amounts of text that continues a prompt with likely text. |
| **Prompt** | The text we send to the model: the instructions, the facts, and the question. |
| **Baseline** | A simple reference result that a more advanced system must beat. |

## Setup
Run this cell once to load the course helpers, then run the steps below in order. Practice answers are scripted. For a live model, choose `PROVIDER` (`"freeinference"`, `"gemini"`, or `"deepseek"`), save its key in Colab Secrets (`FREEINFERENCE_API_KEY`, `GEMINI_API_KEY`, or `DEEPSEEK_API_KEY`), and change `MODE` to `"live"`. `ask` stops after 12 calls; run this cell again to reset the count.

In [ ]:
from importlib.util import find_spec
if not all(find_spec(name) for name in ['coursekit']):
    %pip install -q "agentic-ai-coursekit @ https://github.com/mkassaf/agentic-ai-course/archive/9c78f2181b0f793c60b2b081c79ae74276026e7d.zip"
from coursekit import make_ask
MODE = "practice"
PROVIDER = "freeinference"  # or "gemini" or "deepseek"
ask = make_ask(mode=MODE, provider=PROVIDER)

## Step 1 · Write the policy as a function

A *project* is a dictionary of facts. `decide` gives the two facts names first (`big_team` and `independent`), so each `if` reads like a line of the policy. It returns two values, the decision and the reason. The line `decision, reason = decide(portal)` unpacks them into two variables.

In [ ]:
def decide(project):
    """Apply the course policy. Return the decision and the reason for it."""
    big_team = project["team_size"] >= 8
    independent = project["independent_deploys"]
    if big_team and independent:
        return "microservices", "8+ people and independent deploys"
    if not big_team:
        return "modular monolith", f"only {project['team_size']} people; the policy needs 8+"
    return "modular monolith", "big team, but the services can share one release"

portal = {"name": "Course portal", "team_size": 3, "independent_deploys": False}
decision, reason = decide(portal)
print(f"{portal['name']}: {decision} ({reason})")

## Step 2 · Run the rule on four projects

The four projects cover every combination of the two facts: neither, both, a big team only, and independent deploys only. A good set of examples covers each path through the code.

In `{project['name']:<18}`, the `:<18` pads the text to 18 characters, so the columns line up.

In [ ]:
projects = [
    {"name": "Course portal",     "team_size": 3,  "independent_deploys": False},
    {"name": "Payments platform", "team_size": 12, "independent_deploys": True},
    {"name": "Internal wiki",     "team_size": 9,  "independent_deploys": False},
    {"name": "Mobile backend",    "team_size": 5,  "independent_deploys": True},
]
for project in projects:
    decision, reason = decide(project)
    print(f"{project['name']:<18} {decision:<17} {reason}")

## Step 3 · Turn a free-text reply into a label

A model replies with text, not with one of our labels. It may write `Modular Monolith.` or add a sentence. `to_label` looks for each label's key word. When a reply mentions both labels or neither, it returns `unclear` instead of guessing. The last test reply shows that this helper is cautious, not clever.

In [ ]:
def to_label(reply):
    """Map a model's free-text reply to one of our two labels, or 'unclear'."""
    text = reply.lower()
    if "monolith" in text and "microservice" not in text:
        return "modular monolith"
    if "microservice" in text and "monolith" not in text:
        return "microservices"
    return "unclear"

for reply in ["Modular Monolith.", "  microservices\n", "Either could work", "Not microservices; a monolith"]:
    print(f"{reply!r:<33} -> {to_label(reply)}")

## Step 4 · Ask a model about the same projects

`ask(prompt, example=...)` sends a prompt and returns the reply as text. In practice mode no model is called, and `ask` returns the scripted `example` reply. In live mode, `example` is ignored.

The prompt does **not** include our policy, so the model must guess. The scripted replies imitate a model that guesses from general habits: a big team or independent deploys "sounds like" microservices. `agreed += answer == expected` adds 1 when the two match, because Python counts `True` as 1.

In [ ]:
# Practice mode returns these scripted guesses. Live mode ignores them and calls the model.
scripted = {"Course portal": "Modular monolith", "Payments platform": "Microservices",
            "Internal wiki": "Microservices.", "Mobile backend": "microservices"}
agreed = 0
for project in projects:
    prompt = f"Should this project use a modular monolith or microservices? {project} Reply with the label only."
    answer = to_label(ask(prompt, example=scripted[project["name"]]))
    expected, _ = decide(project)
    agreed += answer == expected
    print(f"{project['name']:<18} rule: {expected:<17} model: {answer}")
print(f"The model agreed with the rule on {agreed} of {len(projects)} projects.")

## What you should see

Step 2 prints one row per project. Only the Payments platform gets `microservices`, because it is the only project where both facts hold.

Step 4 ends with `The model agreed with the rule on 2 of 4 projects.` The scripted model is wrong on the Internal wiki and the Mobile backend. Those are the mixed cases, where only one fact holds. In live mode your count may differ, because a real model is guessing too. Here, a disagreement is a useful result, not a bug.

## Try it

1. Add a fifth project, `{"name": "Search service", "team_size": 8, "independent_deploys": True}`, to `projects`. Predict its row in Step 2 before you run it. Then add `"Search service": "microservices"` to `scripted` and run Step 4.
2. Change `>= 8` to `> 8` in `decide`. Which rows change? Only the Search service does, which is why later labs test the boundary value 8.
3. Switch to live mode and run Step 4. Does a real model make the same mistakes on the mixed cases?

## Takeaway

The Python rule is exact, and it explains each answer, but it only knows the policy we wrote. A model can answer an open question, but it cannot follow a policy it has never seen. Lab 01 puts the policy into the prompt and turns this comparison into a proper test.

## Read further

- [FreeInference quickstart](https://doc.freeinference.org/quickstart)
- [Prompt engineering](https://developers.openai.com/api/docs/guides/prompt-engineering)